In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Set random seed for consistent results
RNG = np.random.default_rng(42)

# Simulation settings
NUM_ROUTERS = 3
HOURS = 24 * 14
CONGESTION_CAPACITY = 55


# Create simulated traffic data
records = []

for router_id in range(NUM_ROUTERS):
    base_load = 30 + router_id * 10

    for hour in range(HOURS):
        day = hour // 24
        hour_of_day = hour % 24

        # Identify weekends
        is_weekend = 1 if (day % 7) >= 5 else 0

        # Add daily traffic pattern
        daily_pattern = max(
            25 * np.sin((hour_of_day - 6) * np.pi / 12),
            -10
        )

        weekend_effect = -10 if is_weekend else 0

        # Add random variation
        noise = RNG.normal(0, 5)

        traffic = max(
            base_load + daily_pattern + weekend_effect + noise,
            0
        )

        records.append({
            "router_id": router_id,
            "hour": hour,
            "hour_of_day": hour_of_day,
            "day_of_week": day % 7,
            "is_weekend": is_weekend,
            "traffic_volume": traffic
        })


# Convert data into a dataframe
df = pd.DataFrame(records)

df = df.sort_values(
    ["router_id", "hour"]
).reset_index(drop=True)


# Add previous traffic values as features
df["lag_1h"] = df.groupby("router_id")["traffic_volume"].shift(1)
df["lag_24h"] = df.groupby("router_id")["traffic_volume"].shift(24)

# Remove rows without previous traffic values
df = df.dropna().reset_index(drop=True)


# Select input features and target value
feature_cols = [
    "router_id",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "lag_1h",
    "lag_24h"
]

X = df[feature_cols]
y = df["traffic_volume"]


# Split training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)


# Train different prediction models
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=150,
        random_state=42
    )
}


# Compare model performance
for name, model in models.items():
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    print(
        f"{name}: "
        f"MAE={mean_absolute_error(y_test, preds):.3f} "
        f"RMSE={np.sqrt(mean_squared_error(y_test, preds)):.3f} "
        f"R2={r2_score(y_test, preds):.4f}"
    )


# Use the better model for congestion prediction
best_model = models["RandomForest"]

preds = best_model.predict(X_test)


# Create routing decisions based on predictions
decisions = X_test.copy()

decisions["predicted_traffic"] = preds
decisions["actual_traffic"] = y_test.values

decisions["predicted_congestion"] = (
    decisions["predicted_traffic"] > CONGESTION_CAPACITY
)

decisions["actual_congestion"] = (
    decisions["actual_traffic"] > CONGESTION_CAPACITY
)

decisions["routing_action"] = decisions[
    "predicted_congestion"
].apply(
    lambda c: "REROUTE" if c else "NORMAL"
)


# Calculate prediction accuracy for congestion detection
tp = (
    (decisions.predicted_congestion)
    & (decisions.actual_congestion)
).sum()

fp = (
    (decisions.predicted_congestion)
    & (~decisions.actual_congestion)
).sum()

fn = (
    (~decisions.predicted_congestion)
    & (decisions.actual_congestion)
).sum()


precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0


print(f"\nPrecision: {precision:.3f}  Recall: {recall:.3f}")

# Show sample routing decisions
print(
    decisions[
        [
            "router_id",
            "hour_of_day",
            "predicted_traffic",
            "actual_traffic",
            "routing_action"
        ]
    ].head(15)
)

LinearRegression: MAE=5.414 RMSE=6.883 R2=0.7811
RandomForest: MAE=5.291 RMSE=6.700 R2=0.7926

Precision: 0.910  Recall: 0.803
     router_id  hour_of_day  predicted_traffic  actual_traffic routing_action
748          2            4          29.297330       27.593794         NORMAL
749          2            5          29.572695       40.718409         NORMAL
750          2            6          42.468865       34.189602         NORMAL
751          2            7          39.758543       35.887204         NORMAL
752          2            8          42.088361       43.190774         NORMAL
753          2            9          48.774300       57.823219         NORMAL
754          2           10          60.277696       61.805221        REROUTE
755          2           11          59.914414       63.560091        REROUTE
756          2           12          67.570195       71.070949        REROUTE
757          2           13          67.450775       50.783973        REROUTE
758          2 